# VRSBench Image Captioning — Qwen2.5-VL-7B + QLoRA SFT (Metric-Aware)
Same length-control + metric-aware prompting approach as the 2B run, scaled to 7B.
Scoped tightly for a 1-1.5hr budget: 2.5K train images, 2 epochs, 30-image val.
Few-shot examples are used at INFERENCE only (not training) — at 7B scale, prepending 2 extra
images to every training sample roughly triples per-step image-token cost, which doesn't fit
the time budget. SFT itself teaches the target style directly from labels, so few-shot's job
at training time is mostly redundant; it earns its cost back at inference where it's a one-time cost.

In [10]:
!pip install -q "unsloth[colab-new]" trl peft bitsandbytes accelerate
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap


In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # force single-GPU: T4x2 naive model-parallel split was causing
                                            # heavy cross-device sync (~50s/step) — 7B fits on one T4 in 4-bit

import json
import random
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

RUN_START = time.time()
def elapsed_min():
    return (time.time() - RUN_START) / 60


Device: cuda


## 1. Load train captions and build DataFrame

In [12]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)

# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()


(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 2. Train subset — 2.5K images, tightly scoped for 7B at a 1-1.5hr budget

In [13]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(df, test_size=0.1, random_state=SEED)

TRAIN_SUBSET = 1200   # tightened from 2500 — T4 dual-GPU split was killing throughput; even single-GPU,
                       # 7B is slower than 2B per step, so scope stays conservative for the 1.5hr ceiling
train_df = train_df.sample(n=min(TRAIN_SUBSET, len(train_df)), random_state=SEED).reset_index(drop=True)

# held-out fixed examples for the INFERENCE-time few-shot prompt (not used in training — see note above)
FEWSHOT_IDX = [0, 1]
fewshot_examples = train_df.iloc[FEWSHOT_IDX].reset_index(drop=True)
train_df = train_df.drop(index=FEWSHOT_IDX).reset_index(drop=True)

print("Train:", len(train_df), "| Few-shot (inference-only) examples:", len(fewshot_examples))


Train: 1198 | Few-shot (inference-only) examples: 2


## 3. Load Qwen2.5-VL-7B 4-bit + LoRA (Unsloth)
Rank 16 / alpha 16 — report's best stability/overfitting trade-off, and the exact config
that achieved 0.6841 BERT-BLEU4 with this same base model + SFT + metric-aware prompting.

In [14]:
from unsloth import FastVisionModel

MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    random_state=SEED,
)


==((====))==  Unsloth 2026.9.11: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 6.330 GiB
no_split classes   : ['Qwen2_5_VLDecoderLayer', 'Qwen2_5_VLVisionBlock']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 5.945 GiB requested
  cuda:0  budget   9.19 GiB  weights  3.165 GiB  free  6.025 GiB  reserve  5.945 GiB
  cuda:1  budget   9.39 GiB  weights  3.165 GiB  free  6.223 GiB  reserve  5.864 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 10.212 -> 9.191 GiB, cuda:1 10.430 -> 9.387 GiB (memory the quantiser 

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

## 4. Metric-aware prompt (+ inference-only few-shot)
Same explicit length cap and anti-hedging instruction as the 2B run.

In [15]:
SYSTEM_PROMPT = (
    "You are a remote-sensing image captioning assistant. "
    "Write EXACTLY one paragraph of 40-55 words describing the satellite/aerial image: "
    "key objects, their approximate spatial layout (e.g. top-left, bottom-right, center), "
    "and scene context. Never exceed 55 words. Do not hedge, apologize, or comment on image "
    "quality/resolution limits — describe only what is visible, plainly and directly. "
    "Do not repeat phrases."
)

def build_fewshot_prefix():
    turns = []
    for _, row in fewshot_examples.iterrows():
        ex_image = Image.open(row["image_path"]).convert("RGB")
        turns.append({"role": "user", "content": [
            {"type": "image", "image": ex_image},
            {"type": "text", "text": "Caption this satellite image."},
        ]})
        turns.append({"role": "assistant", "content": [{"type": "text", "text": row["caption"]}]})
    return turns

FEWSHOT_PREFIX = build_fewshot_prefix()

def build_conversation(image, caption=None, use_fewshot=False):
    """use_fewshot=False during training (cost), True at inference (quality)."""
    user_content = [
        {"type": "image", "image": image},
        {"type": "text", "text": "Caption this satellite image."},
    ]
    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]
    if use_fewshot:
        messages += FEWSHOT_PREFIX
    messages.append({"role": "user", "content": user_content})
    if caption is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": caption}]})
    return messages


## 5. PyTorch Dataset
Training captions truncated to the word budget (length control, layer 1). No few-shot in training data.

In [16]:
from torch.utils.data import Dataset

class VRSBenchVLMDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]

        words = caption.split()
        if len(words) > 55:
            caption = " ".join(words[:55])

        messages = build_conversation(image, caption, use_fewshot=False)
        return {"messages": messages}


train_dataset = VRSBenchVLMDataset(train_df)
print(len(train_dataset))


1198


## 6. Length-control loss (layer 2)
Small auxiliary penalty on assistant response token count, same as the 2B run.

In [17]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

LENGTH_PENALTY_LAMBDA = 0.01

class LengthControlledSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        labels = inputs.get("labels")
        if labels is not None:
            response_lengths = (labels != -100).sum(dim=-1).float()
            length_penalty = response_lengths.mean() * LENGTH_PENALTY_LAMBDA / labels.shape[-1]
            loss = ce_loss + length_penalty
        else:
            loss = ce_loss

        return (loss, outputs) if return_outputs else loss


## 7. Training — 2 epochs, memory-safe batch shape for 7B
Per-device batch dropped to 1 (7B is heavier than 2B even in 4-bit); grad accumulation raised to 16
to keep the same effective batch size of 16.

In [18]:
FastVisionModel.for_training(model)

use_bf16 = torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen7b_vrsbench_lora",
    per_device_train_batch_size=1,      # 7B + 4-bit is memory-tight even with grad checkpointing
    gradient_accumulation_steps=16,     # effective batch size 16, same as the report/2B run
    num_train_epochs=1,      # tightened from 2 — see TRAIN_SUBSET note above; rerun with more once timing confirmed
    learning_rate=2e-4,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=20,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=256,   # no few-shot prefix in training data -> short sequences, keep this tight
    report_to="none",
)

trainer = LengthControlledSFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    args=training_args,
)

train_result = trainer.train()
print(train_result)
print(f"Training took {elapsed_min():.1f} min")


Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 1,198 | Num Epochs = 1 | Total steps = 75
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)


Step,Training Loss
20,112.222693
40,47.775214
60,13.960097


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen7b_vrsbench_lora/checkpoint-75/tokenizer_config.json.


TrainOutput(global_step=75, training_loss=48.24523640950521, metrics={'train_runtime': 3555.9636, 'train_samples_per_second': 0.337, 'train_steps_per_second': 0.021, 'total_flos': 2.759707121676288e+16, 'train_loss': 48.24523640950521, 'epoch': 1.0})
Training took 60.3 min


## 8. Metric suite

In [19]:
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()

    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]

    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0

    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)

    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)])
        idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)])
        idx += len(grams)

    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = sims.max(dim=1).values.mean().item()
            p_n = max(p_n, 1e-9)
        log_p_sum += np.log(p_n)

    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }


## 9. Load official val split, subsample to 30 images

In [21]:
VAL_IMG_DIR = os.path.join(DATA_DIR, "Images_val", "Images_val")
EVAL_CAP_JSON = os.path.join(DATA_DIR, "VRSBench_EVAL_Cap.json")

with open(EVAL_CAP_JSON, "r") as f:
    eval_raw = json.load(f)

eval_df = pd.DataFrame(eval_raw)
if "type" in eval_df.columns:
    eval_df = eval_df[eval_df["type"] == "caption"].reset_index(drop=True)

eval_rename = {}
for c in eval_df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        eval_rename[c] = "filename"
    elif lc in ("image_path", "path"):
        eval_rename[c] = "filename"
    elif "caption" in lc or lc in ("ground_truth", "answer"):
        eval_rename[c] = "caption"

eval_df = eval_df.rename(columns=eval_rename)
eval_df["image_path"] = eval_df["filename"].apply(lambda x: os.path.join(VAL_IMG_DIR, os.path.basename(str(x))))
eval_df = eval_df[["image_path", "caption"]].dropna().reset_index(drop=True)

EVAL_SUBSET_SIZE = 30
eval_subset_df = eval_df.sample(n=min(EVAL_SUBSET_SIZE, len(eval_df)), random_state=SEED).reset_index(drop=True)
print("Eval samples used:", len(eval_subset_df))


Eval samples used: 30


## 10. Inference — few-shot ON here, length-capped, deterministic

In [22]:
FastVisionModel.for_inference(model)

@torch.no_grad()
def generate_caption(image, max_new_tokens=80, min_new_tokens=25):
    messages = build_conversation(image, caption=None, use_fewshot=True)
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(DEVICE)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        use_cache=True,
    )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text.strip()


def validate(eval_df):
    preds, refs = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="val"):
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    metrics = compute_all_metrics(preds, refs)
    return metrics, preds, refs


metrics, preds, refs = validate(eval_subset_df)
print(metrics)
print(f"Total run time: {elapsed_min():.1f} min")


val:   0%|          | 0/30 [00:00<?, ?it/s]

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

{'BLEU-1': 0.11636742300186483, 'BLEU-2': 0.044574690385167065, 'BLEU-3': 0.022623167293373683, 'BLEU-4': 0.013625774252700142, 'METEOR': 0.14226622925916302, 'ROUGE_L': 0.1288735992513335, 'CIDEr': 0.06229477928342549, 'Avg_L': 64.8, 'BERT-BLEU4': 0.4363770790032435}
Total run time: 68.7 min


## 11. Sample predictions

In [23]:
for p, r in list(zip(preds, refs))[:5]:
    print("PRED:", p)
    print("REF :", r)
    print("-" * 60)


PRED: A large ship docked at port near several smaller ships can be seen within high-resolution imagery obtained via Google Earth. There's also another vessel positioned further away but closer than other vessels around it; its size relative compared others suggests that these two may have different sizes despite being close together visually due to perspective distortion inherent when viewing such images through satellites like those used for Google Earth imaging services.
REF : The image from GoogleEarth shows a docking area with a ship. The ship has a white and green color scheme and is very long.
------------------------------------------------------------
PRED: This high-resolution image captured via Google Earth features two basketball courts positioned at different angles within close proximity; both have distinct markings for gameplay areas including circles around free throw lines as well as full court boundaries marked out clearly against each other's surroundings which inclu

## 12. Save LoRA adapter

In [24]:
model.save_pretrained("/kaggle/working/qwen7b_vrsbench_lora_final")
processor.save_pretrained("/kaggle/working/qwen7b_vrsbench_lora_final")
print("Saved.")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen7b_vrsbench_lora_final/tokenizer_config.json.


Saved.
